# Aggregate the Results for a set of Sweeps

In [ ]:
import itertools
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from notebook_utils import parse_tensorboard_logs, aggregate_test_metrics

SWEEPS_DIR = Path("/project/ultrabench2/20260906_results/runs")

sns.set_theme(style="whitegrid", context="paper", font_scale=0.75)

MODEL_DISPLAY_NAMES = {
    "deblurmim_convmae": "Deblurring ConvMAE",
    "deblurmim_mae": "Deblurring MAE",
    "dinov3_convnext_base": "DINOv3 ConvNeXt-B",
    "dinov3_convnext_large": "DINOv3 ConvNeXt-L",
    "dinov3_vitb16": "DINOv3 ViT-B",
    "dinov3_vitl16": "DINOv3 ViT-L",
    "echocare": "EchoCare",
    "openus": "OpenUS",
    "samus": "SAMUS",
    "tinyusfm": "TinyUSFM",
    "ultrafedfm": "UltraFedFM",
    "ultrasam": "UltraSAM",
    "urfm_base": "URFM-B",
    "urfm_large": "URFM-L",
    "usfm": "USFM",
    "usfmae": "USF-MAE",
}


# Read the lists of tasks, models, and modes from the config
TASKS = [
    p.stem
    for p in Path("../src/ultrabench/training/config/task").iterdir()
    if p.suffix == ".yaml"
]

MODELS = [
    p.stem
    for p in Path("../src/ultrabench/training/config/model").iterdir()
    if p.suffix == ".yaml"
]

MODES = [
    p.stem
    for p in Path("../src/ultrabench/training/config/mode").iterdir()
    if p.suffix == ".yaml"
]

SHOT_PERCENTAGES = [1.0]
SEEDS = [42, 43, 44, 45, 46]

print(f"Found {len(TASKS)} tasks: {TASKS}")
print(f"Found {len(MODELS)} models: {MODELS}")
print(f"Found {len(MODES)} modes: {MODES}")

## Parse Tensorboard Logs

In [ ]:
results = parse_tensorboard_logs(SWEEPS_DIR)

# Update URFM Base model name
results["model"] = results["model"].replace("urfm", "urfm_base")

# Filter tasks, models, modes, and shot percentages
results = results[
    (
        results["mode"].str.endswith("_head_only")
        | results["mode"].str.endswith("_full_finetuning")
    )
    & results["task"].isin(TASKS)
    & results["model"].isin(MODELS)
    & results["shot_pct"].isin(SHOT_PERCENTAGES)
    & results["seed"].isin(SEEDS)
]

# Keep only the last run for each combination of model, task, shot percentage,
# mode, and seed
results = (
    results.sort_values("datetime")
    .groupby(["model", "task", "shot_pct", "mode", "seed"], as_index=False)
    .last()
)

results.to_csv(f"trainval_results_{datetime.now().strftime('%Y%m%d')}.csv", index=False)

print(f"Number of runs after filtering: {len(results)}")
results.head()

In [ ]:
# Check that there are runs for every combination of model, task, mode,
# shot_pct, and seed

for task, model, mode, shot_pct, seed in itertools.product(
    TASKS, MODELS, MODES, SHOT_PERCENTAGES, SEEDS
):
    if not (
        (results["task"] == task)
        & (results["mode"].str.endswith(mode))
        & (results["model"] == model)
        & (results["shot_pct"] == shot_pct)
        & (results["seed"] == seed)
    ).any():
        print(f"Missing run for {model}, {task}, {shot_pct}, {mode}, seed {seed}")

In [ ]:
classification_results = results[results["dir_name"].str.contains("classification")]
segmentation_results = results[results["dir_name"].str.contains("segmentation")]

In [ ]:
# Group results by dir_name and keep only the rows with the lowest
# `Epoch/val_loss` value for each group
classification_results = (
    classification_results.sort_values("val_loss")
    .groupby("dir_name")
    .first()
    .reset_index()
)

segmentation_results = (
    segmentation_results.sort_values("val_loss")
    .groupby("dir_name")
    .first()
    .reset_index()
)

## Calculate Best Validation Statistics Across Runs

In [ ]:
# Group the classification results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_acc`
# for each group
classification_summary = (
    classification_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "val_loss": ["mean", "std"],
            "val_acc": ["mean", "std"],
        }
    )
    .reset_index()
)
classification_summary.round(4)

In [ ]:
# Group the segmentation results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_dice`
# for each group
segmentation_summary = (
    segmentation_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "val_loss": ["mean", "std"],
            "val_dice": ["mean", "std"],
        }
    )
    .reset_index()
)
segmentation_summary.round(4)

## Calculate Test Metric Statistics

In [ ]:
test_results = aggregate_test_metrics(SWEEPS_DIR)

# Filter tasks, models, modes, and shot percentages
test_results = test_results[
    (
        test_results["mode"].str.endswith("_head_only")
        | test_results["mode"].str.endswith("_full_finetuning")
    )
    & test_results["task"].isin(TASKS)
    & test_results["model"].isin(MODELS)
    & test_results["shot_pct"].isin(SHOT_PERCENTAGES)
    & test_results["seed"].isin(SEEDS)
]

# Keep only the last run for each combination of model, task, shot percentage,
# mode, and seed
test_results = (
    test_results.sort_values("datetime")
    .groupby(["model", "task", "shot_pct", "mode", "seed"], as_index=False)
    .last()
)

test_results.to_csv(
    f"test_results_{datetime.now().strftime('%Y%m%d')}.csv", index=False
)

print(f"Number of runs after filtering: {len(test_results)}")
test_results.head()

In [ ]:
# Check that there are runs for every combination of model, task, mode,
# shot_pct, and seed

for task, model, mode, shot_pct, seed in itertools.product(
    TASKS, MODELS, MODES, SHOT_PERCENTAGES, SEEDS
):
    if not (
        (test_results["task"] == task)
        & (test_results["mode"].str.endswith(mode))
        & (test_results["model"] == model)
        & (test_results["shot_pct"] == shot_pct)
        & (test_results["seed"] == seed)
    ).any():
        print(f"Missing run for {model}, {task}, {shot_pct}, {mode}, seed {seed}")

In [ ]:
# Group the classification results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_acc`
# for each group
classification_test_results = test_results[
    test_results["task"].str.contains("classification")
]

classification_test_results["accuracy"] = classification_test_results.apply(
    lambda row: (
        row["seq_accuracy"] if not pd.isna(row["seq_accuracy"]) else row["accuracy"]
    ),
    axis=1,
)
classification_test_results["balanced_accuracy"] = classification_test_results.apply(
    lambda row: (
        row["seq_balanced_accuracy"]
        if not pd.isna(row["seq_balanced_accuracy"])
        else row["balanced_accuracy"]
    ),
    axis=1,
)
classification_test_results["f1_macro"] = classification_test_results.apply(
    lambda row: (
        row["seq_f1_macro"] if not pd.isna(row["seq_f1_macro"]) else row["f1_macro"]
    ),
    axis=1,
)
classification_test_results["f1_weighted"] = classification_test_results.apply(
    lambda row: (
        row["seq_f1_weighted"]
        if not pd.isna(row["seq_f1_weighted"])
        else row["f1_weighted"]
    ),
    axis=1,
)
classification_test_results["mcc"] = classification_test_results.apply(
    lambda row: row["seq_mcc"] if not pd.isna(row["seq_mcc"]) else row["mcc"],
    axis=1,
)

classification_test_summary = (
    classification_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "accuracy": ["mean", "std"],
            "balanced_accuracy": ["mean", "std"],
            "f1_macro": ["mean", "std"],
            "f1_weighted": ["mean", "std"],
            "mcc": ["mean", "std"],
        }
    )
    .reset_index()
)
classification_test_summary.round(4)

In [ ]:
# Group the segmentation results by `model`, `task`, `shot_pct` and `mode`
# and compute the mean and std. dev. of `Epoch/val_loss` and `Epoch/val_dice`
# for each group
segmentation_test_results = test_results[
    test_results["task"].str.contains("segmentation")
]

segmentation_test_summary = (
    segmentation_test_results.groupby(["task", "model", "shot_pct", "mode"])
    .agg(
        {
            "dice": ["mean", "std"],
            "positive_dice": ["mean", "std"],
            "iou": ["mean", "std"],
        }
    )
    .reset_index()
)
segmentation_test_summary.round(4)

In [ ]:
segmentation_test_results[
    segmentation_test_results["mode"].str.contains("full_finetuning")
].groupby(["task"]).agg(
    {
        "dice": ["mean", "std"],
        "positive_dice": ["mean", "std"],
        "iou": ["mean", "std"],
    }
)

In [ ]:
# Plot the linear probing classification test results
x_tick_labels = [
    "AUL",
    "BUS-BRA\n(Hist.)",
    "BUS-BRA\n(Path.)",
    "CAMUS",
    "COVID-BLUES",
    "Fetal Planes\n(Brain)",
    "Fetal Planes\n(Fetus)",
    "GBCU",
    "MASLD",
    "MMOTU",
    "TN3K",
]

g = sns.catplot(
    data=classification_test_results[
        classification_test_results["mode"].str.contains("head_only")
    ],
    x="task",
    y="balanced_accuracy",
    errorbar="sd",
    hue="model",
    kind="bar",
    order=sorted(classification_test_results["task"].unique()),
    height=2.75,
    aspect=2,
)
g.set_axis_labels("Task", "Balanced Accuracy")
g.set_xticklabels(x_tick_labels, rotation=90)

# Replace the model names in the legend with the display names
for t in g._legend.texts:
    t.set_text(MODEL_DISPLAY_NAMES[t.get_text()])

g.ax.set_ylim(0, 1)

plt.savefig("classification_probing_test_results.pdf", bbox_inches="tight")
plt.savefig("classification_probing_test_results.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Plot the full fine-tuning classification test results
x_tick_labels = [
    "AUL",
    "BUS-BRA\n(Hist.)",
    "BUS-BRA\n(Path.)",
    "CAMUS",
    "COVID-BLUES",
    "Fetal Planes\n(Brain)",
    "Fetal Planes\n(Fetus)",
    "GBCU",
    "MASLD",
    "MMOTU",
    "TN3K",
]

g = sns.catplot(
    data=classification_test_results[
        classification_test_results["mode"].str.contains("full_finetuning")
    ],
    x="task",
    y="balanced_accuracy",
    errorbar="sd",
    hue="model",
    kind="bar",
    order=sorted(classification_test_results["task"].unique()),
    height=2.75,
    aspect=2,
)
g.set_axis_labels("Task", "Balanced Accuracy")
g.set_xticklabels(x_tick_labels, rotation=90)

# Replace the model names in the legend with the display names
for t in g._legend.texts:
    t.set_text(MODEL_DISPLAY_NAMES[t.get_text()])

g.ax.set_ylim(0, 1)

plt.savefig("classification_fft_test_results.pdf", bbox_inches="tight")
plt.savefig("classification_fft_test_results.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Plot the upsampler probing segmentation test results
x_tick_labels = [
    "AUL\n(Liver)",
    "AUL\n(Mass)",
    "BUS-BRA",
    "CAMUS",
    "MMOTU",
    "Open Kidney\n(Capsule)",
    "Open Kidney\n(Regions)",
    "PSFHS",
    "STMUS",
    "TN3K",
]

g = sns.catplot(
    data=segmentation_test_results[
        segmentation_test_results["mode"].str.contains("head_only")
    ],
    x="task",
    y="positive_dice",
    errorbar="sd",
    hue="model",
    kind="bar",
    order=sorted(segmentation_test_results["task"].unique()),
    height=2.75,
    aspect=2,
)
g.set_axis_labels("Task", "Positive Dice Score")
g.set_xticklabels(x_tick_labels, rotation=90)

# Replace the model names in the legend with the display names
for t in g._legend.texts:
    t.set_text(MODEL_DISPLAY_NAMES[t.get_text()])

plt.savefig("segmentation_probing_test_results.pdf", bbox_inches="tight")
plt.savefig("segmentation_probing_test_results.png", bbox_inches="tight", dpi=300)
plt.show()

In [ ]:
# Plot the full fine-tuning (upsampler) segmentation test results
x_tick_labels = [
    "AUL\n(Liver)",
    "AUL\n(Mass)",
    "BUS-BRA",
    "CAMUS",
    "MMOTU",
    "Open Kidney\n(Capsule)",
    "Open Kidney\n(Regions)",
    "PSFHS",
    "STMUS",
    "TN3K",
]

g = sns.catplot(
    data=segmentation_test_results[
        segmentation_test_results["mode"].str.contains("full_finetuning")
    ],
    x="task",
    y="positive_dice",
    errorbar="sd",
    hue="model",
    kind="bar",
    order=sorted(segmentation_test_results["task"].unique()),
    height=2.75,
    aspect=2,
)
g.set_axis_labels("Task", "Positive Dice Score")
g.set_xticklabels(x_tick_labels, rotation=90)

# Replace the model names in the legend with the display names
for t in g._legend.texts:
    t.set_text(MODEL_DISPLAY_NAMES[t.get_text()])

plt.savefig("segmentation_fft_test_results.pdf", bbox_inches="tight")
plt.savefig("segmentation_fft_test_results.png", bbox_inches="tight", dpi=300)
plt.show()